# Chinese Rap Lyrics NER Pipeline

端到端流水线：数据清洗 → NER 实体识别 → Bag-of-Entities → K-Means 聚类

直接 **Run All** 即可看到完整结果。

In [ ]:
import sys
from pathlib import Path

# ensure project root is on path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from collections import Counter

## 1. 数据加载与清洗

清洗步骤：
- 移除 Live 版本、伴奏/instrumental 等非原创录音室版本
- 从歌词文本中剥离制作人信息行（出品、Prod.、混音、母带等）
- 移除独立的结构标记行（Verse/Hook/Chorus 等标签行）

In [ ]:
from src.data_cleaning import load_data, clean_songs, clean_text, combine_by_artist

# 加载原始数据
df_raw = load_data("lyrics_chunks_enriched.csv")
print(f"原始数据: {len(df_raw)} 行, {df_raw['artist'].nunique()} 位艺人, {df_raw['song_id'].nunique()} 首歌")

In [ ]:
# 移除 Live/伴奏 等非原创录音室版本
df_cleaned = clean_songs(df_raw)
print(f"\n清洗后: {len(df_cleaned)} 行, {df_cleaned['song_id'].nunique()} 首歌")

In [ ]:
# 清理歌词文本（移除制作信息行、结构标记行）
df_cleaned = clean_text(df_cleaned)
print(f"文本清洗后: {len(df_cleaned)} 行")

# 查看清洗前后对比
print(f"\n--- 清洗总结 ---")
print(f"原始行数:   {len(df_raw)}")
print(f"清洗后行数: {len(df_cleaned)}")
print(f"移除比例:   {(len(df_raw) - len(df_cleaned)) / len(df_raw) * 100:.1f}%")

In [ ]:
# 按艺人合并歌词
artist_lyrics = combine_by_artist(df_cleaned)
print(f"合并后: {len(artist_lyrics)} 位艺人")
print(f"\n歌词长度分布:")
artist_lyrics["text_len"] = artist_lyrics["combined_text"].str.len()
print(artist_lyrics["text_len"].describe())

# 显示前 10 位艺人和歌词长度
top10 = artist_lyrics.nlargest(10, "text_len")[["artist", "text_len"]]
print(f"\n歌词最长的 10 位艺人:")
for _, r in top10.iterrows():
    print(f"  {r['artist']}: {r['text_len']:,} 字符")

## 2. NER 实体识别

In [ ]:
from src.ner import build_nlp, extract_entities, entity_summary

nlp = build_nlp("configs/rap_lexicon_seed.jsonl")

In [ ]:
# 提取所有艺人的命名实体
entity_df = extract_entities(artist_lyrics, nlp, min_entity_len=2)

In [ ]:
# Top 30 全局高频实体
print("=== Top 30 高频实体 ===")
top30 = entity_summary(entity_df, top_n=30)
print(top30.to_string(index=False))

In [ ]:
# 每个标签类型的实体数量
print("=== 标签分布 ===")
label_dist = entity_df["label"].value_counts()
print(label_dist)

# 每种标签的 top-5 实体
print("\n=== 各标签类型 Top-5 实体 ===")
for label in label_dist.index:
    subset = entity_df[entity_df["label"] == label]
    top5 = subset["entity"].value_counts().head(5)
    print(f"\n[{label}] ({len(subset)} mentions)")
    for ent, cnt in top5.items():
        print(f"  {ent}: {cnt}")

## 3. Bag-of-Entities 矩阵与聚类

In [ ]:
from src.clustering import build_bag_of_entities, run_kmeans, summarize_clusters

entity_matrix = build_bag_of_entities(entity_df)
print(f"Bag-of-Entities 矩阵: {entity_matrix.shape[0]} 艺人 × {entity_matrix.shape[1]} 实体")
print(f"稀疏度: {(entity_matrix == 0).sum().sum() / entity_matrix.size * 100:.1f}%")

In [ ]:
# K-Means 聚类
N_CLUSTERS = 6

assignments, centroids = run_kmeans(entity_matrix, n_clusters=N_CLUSTERS, random_state=42)

print(f"=== 聚类结果（K={N_CLUSTERS}）===")
print(f"\n各聚类的艺人数量:")
print(assignments["cluster"].value_counts().sort_index())

In [ ]:
# 查看每个聚类的艺人
print("=== 各聚类艺人列表 ===")
for cluster_id in sorted(assignments["cluster"].unique()):
    artists_in_cluster = assignments[assignments["cluster"] == cluster_id]["artist"].tolist()
    print(f"\n--- Cluster {cluster_id} ({len(artists_in_cluster)} artists) ---")
    print(", ".join(artists_in_cluster))

In [ ]:
# 聚类摘要：每个聚类的代表实体
summaries = summarize_clusters(centroids, top_k=15)

print("=== 各聚类代表实体（Top 15）===")
for cluster_name in summaries["cluster"].unique():
    cluster_data = summaries[summaries["cluster"] == cluster_name]
    # 只显示权重 > 0 的
    cluster_data = cluster_data[cluster_data["centroid_weight"] > 0]
    print(f"\n--- {cluster_name} ---")
    for _, r in cluster_data.iterrows():
        print(f"  {r['entity']}: {r['centroid_weight']:.2f}")

## 4. 保存结果

In [ ]:
from src.io_utils import save_outputs

save_outputs(
    output_dir="outputs",
    artist_lyrics=artist_lyrics.drop(columns=["text_len"], errors="ignore"),
    entity_df=entity_df,
    entity_matrix=entity_matrix,
    assignments=assignments,
    summaries=summaries,
)

## 5. 质量检查

抽样检查实体识别的效果，帮助发现 false positive / false negative。

In [ ]:
# 单个艺人的实体详细检查
CHECK_ARTIST = artist_lyrics.iloc[0]["artist"]
print(f"=== 质量检查: {CHECK_ARTIST} ===")

artist_entities = entity_df[entity_df["artist"] == CHECK_ARTIST]
print(f"该艺人共提取 {len(artist_entities)} 个实体 mention")
print(f"\n实体频率:")
for (ent, label), cnt in artist_entities.groupby(["entity", "label"]).size().sort_values(ascending=False).head(20).items():
    print(f"  {ent} ({label}): {cnt}")

# 显示该艺人歌词片段以便人工核验
text_sample = artist_lyrics[artist_lyrics["artist"] == CHECK_ARTIST]["combined_text"].iloc[0]
print(f"\n歌词片段（前 500 字）:")
print(text_sample[:500])

In [ ]:
# 可疑实体检查：找出可能是噪声的实体
print("=== 可疑实体（可能是噪声）===")

# 1. 只出现 1 次的实体
entity_counts = entity_df["entity"].value_counts()
singletons = entity_counts[entity_counts == 1]
print(f"\n只出现 1 次的实体数: {len(singletons)} / {len(entity_counts)} ({len(singletons)/len(entity_counts)*100:.1f}%)")
print("样本:", singletons.head(20).index.tolist())

# 2. 实体文本长度异常（太长可能是误识别）
long_entities = entity_df[entity_df["entity"].str.len() > 10]["entity"].unique()
print(f"\n长度 > 10 的实体 ({len(long_entities)} 个):")
for e in long_entities[:20]:
    print(f"  \"{e}\"")